In [ ]:
import pandas as pd
import numpy as np
import ast
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torchdiffeq import odeint
from sklearn.metrics import roc_auc_score, average_precision_score
from torch.optim.lr_scheduler import ReduceLROnPlateau
import time
import os
import warnings
import pickle
import json
from torch_geometric.nn import GATConv
from datetime import datetime

warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# GPU対応
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# ==========================================
# 1. パス設定 
# ==========================================
DATA_PATH =  "/home/nakamuraroi/kumagai/"

# データ前処理
def preprocess_data():
    print("データ前処理を開始...")
    try:
        # パス結合をOSに依存しない形にしつつ、指定された構成に合わせる
        kg_path = os.path.join(DATA_PATH, "work/dataset/kg_dataset.csv")
        kumagai_path = os.path.join(DATA_PATH, "work/dataset/kumagai_patentdata.csv")
        
        print(f"読み込みパス: {kg_path}")
        kg_df = pd.read_csv(kg_path)
        kumagai_df = pd.read_csv(kumagai_path)
    except FileNotFoundError as e:
        print(f"エラー: データファイルが見つかりません。{e}")
        return None

    merged_df = pd.merge(kg_df, kumagai_df, on="patent_number", how="left")
    cleaned_df = merged_df.dropna(subset=["corporation", "patent_number"], how="any")

    cols = [f"pca_text_dim_{i}" for i in range(64)] + [f"node_dim_{i}" for i in range(64)]
    cleaned_df[cols] = cleaned_df[cols].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
    cleaned_df[cols] = cleaned_df[cols].astype(np.float32)

    def safe_literal_eval(s):
        try:
            if isinstance(s, list):
                return s
            return ast.literal_eval(s)
        except (ValueError, SyntaxError):
            return []

    cleaned_df["corporation"] = cleaned_df["corporation"].apply(safe_literal_eval)
    cleaned_df['year_month'] = pd.to_datetime(cleaned_df['year_month'], format='%Y-%m')

    start_date = '2000-01-01'
    end_date = '2023-12-31'

    cleaned_df = cleaned_df[
        (cleaned_df['year_month'] >= start_date) &
        (cleaned_df['year_month'] <= end_date)
    ].copy()
    print("✓ データ前処理完了")
    return cleaned_df

def build_global_graphs(cleaned_df):
    if cleaned_df is None:
        return None, None, None, None, None, None

    year_groups = cleaned_df.groupby(cleaned_df['year_month'].dt.year)
    all_corporations = set()
    all_patents = set()

    for year, group in year_groups:
        for corps in group['corporation']:
            all_corporations.update(corps)
        all_patents.update(group['patent_number'].unique())

    all_corporations = sorted(list(all_corporations))
    all_patents = sorted(list(all_patents))

    global_corp_to_idx = {corp: idx for idx, corp in enumerate(all_corporations)}
    global_patent_to_idx = {patent: idx + len(all_corporations) for idx, patent in enumerate(all_patents)}
    total_global_nodes = len(all_corporations) + len(all_patents)

    print(f"グラフ構築: 企業数={len(all_corporations)}, 特許数={len(all_patents)}")

    print("特許特徴量を準備中...")
    patent_features = {}
    for _, row in cleaned_df.iterrows():
        if row['patent_number'] in global_patent_to_idx:
            pca_feat = row[[f'pca_text_dim_{i}' for i in range(64)]].values.astype(np.float32)
            node_feat = row[[f'node_dim_{i}' for i in range(64)]].values.astype(np.float32)
            patent_features[row['patent_number']] = np.concatenate([pca_feat, node_feat])

    global_graph_dict = {}

    for year, group in year_groups:
        active_nodes = set()
        edges = []

        for _, row in group.iterrows():
            for corp in row['corporation']:
                if corp in global_corp_to_idx and row['patent_number'] in global_patent_to_idx:
                    corp_idx = global_corp_to_idx[corp]
                    patent_idx = global_patent_to_idx[row['patent_number']]
                    edges.append([corp_idx, patent_idx])  # 企業 → 特許
                    edges.append([patent_idx, corp_idx])  # 特許 → 企業
                    active_nodes.add(corp_idx)
                    active_nodes.add(patent_idx)

        if not edges:
            continue

        corp_patent_mapping = {corp: []for corp in all_corporations}
        for _, row in cleaned_df.iterrows():

        for patent_num, patent_idx in global_patent_to_idx.items():
            if patent_num in patent_features:
                x[patent_idx] = torch.tensor(patent_features[patent_num], dtype=torch.float32)

        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        active_mask = torch.zeros(total_global_nodes, dtype=torch.bool)
        active_mask[list(active_nodes)] = True

        global_graph_dict[year] = Data(
            x=x,
            edge_index=edge_index,
            num_nodes=total_global_nodes,
            active_mask=active_mask,
            year=year
        )

    print(f"✓ {len(global_graph_dict)}年分のグラフを構築完了")
    return global_graph_dict, all_corporations, all_patents, total_global_nodes, global_corp_to_idx, global_patent_to_idx

# ModelResultsSaver クラス
class ModelResultsSaver:
    def __init__(self, output_dir="model_outputs"):
        # パス結合を修正
        self.base_output_dir = os.path.join(DATA_PATH, output_dir)
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.experiment_dir = os.path.join(self.base_output_dir, f"experiment_{self.timestamp}")

        os.makedirs(self.experiment_dir, exist_ok=True)
        os.makedirs(os.path.join(self.experiment_dir, "latent_vectors"), exist_ok=True)
        os.makedirs(os.path.join(self.experiment_dir, "future_predictions"), exist_ok=True)
        os.makedirs(os.path.join(self.experiment_dir, "models"), exist_ok=True)
        os.makedirs(os.path.join(self.experiment_dir, "metadata"), exist_ok=True)
        print(f"結果は {self.experiment_dir} に保存されます")

        self.results = {}
        self.metadata = None

    def save_metadata(self, all_corporations, all_patents, global_corp_to_idx, global_patent_to_idx, years, cleaned_df=None):
        metadata = {
            'corporations': all_corporations,
            'patents': all_patents,
            'corp_to_idx': global_corp_to_idx,
            'patent_to_idx': global_patent_to_idx,
            'years': years,
            'timestamp': self.timestamp,
            'num_corporations': len(all_corporations),
            'num_patents': len(all_patents),
            'total_nodes': len(all_corporations) + len(all_patents)
        }
        self.metadata = metadata
        with open(os.path.join(self.experiment_dir, "metadata", "experiment_metadata.json"), 'w', encoding='utf-8') as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
        with open(os.path.join(self.experiment_dir, "metadata", "index_mappings.pkl"), 'wb') as f:
            pickle.dump({
                'corp_to_idx': global_corp_to_idx,
                'patent_to_idx': global_patent_to_idx
            }, f)
        print(f"メタデータを保存: {self.experiment_dir}/metadata/")

    def extract_and_save_latent_vectors(self, model, global_graph_dict, num_corps, model_name, phase="after_training"):
        model.eval()
        phase_dir = os.path.join(self.experiment_dir, "latent_vectors", f"{phase}_{model_name.replace(' ', '_')}")
        os.makedirs(phase_dir, exist_ok=True)
        all_latent_data = {}

        with torch.no_grad():
            for year, data in global_graph_dict.items():
                data = data.to(device)
                node_indices = torch.arange(model.num_nodes, device=device)
                try:
                    x_features = model.get_node_features(data.x, node_indices)
                    z, mu, logvar = model.encode(x_features, data.edge_index)

                    company_mask = torch.arange(model.num_nodes, device=device) < num_corps
                    patent_mask = torch.arange(model.num_nodes, device=device) >= num_corps
                    active_mask = data.active_mask

                    z_companies = z[company_mask & active_mask].cpu().numpy()
                    z_patents = z[patent_mask & active_mask].cpu().numpy()
                    
                    active_company_indices = torch.arange(model.num_nodes, device=device)[company_mask & active_mask].cpu().numpy()
                    active_patent_indices = torch.arange(model.num_nodes, device=device)[patent_mask & active_mask].cpu().numpy()

                    year_data = {
                        'year': year,
                        'company_vectors': {'z': z_companies, 'indices': active_company_indices},
                        'patent_vectors': {'z': z_patents, 'indices': active_patent_indices},
                    }
                    all_latent_data[year] = year_data
                    
                    # CSV保存（可視化用）
                    if len(z_companies) > 0:
                        df_comp = pd.DataFrame(z_companies, columns=[f'latent_dim_{i}' for i in range(z_companies.shape[1])])
                        df_comp.to_csv(os.path.join(phase_dir, f"companies_{year}.csv"), index=False)
                    
                    if len(z_patents) > 0:
                        df_pat = pd.DataFrame(z_patents, columns=[f'latent_dim_{i}' for i in range(z_patents.shape[1])])
                        df_pat.to_csv(os.path.join(phase_dir, f"patents_{year}.csv"), index=False)
                        
                except Exception as e:
                    print(f"年度{year}の潜在ベクトル抽出エラー: {e}")
                    continue
                    
        with open(os.path.join(phase_dir, "all_latent_vectors.pkl"), 'wb') as f:
            pickle.dump(all_latent_data, f)
        print(f"潜在ベクトル保存完了: {phase_dir}")
        return all_latent_data

    def save_future_link_predictions(self, model, global_graph_dict, num_corps, model_name):
        # 簡易実装（詳細な保存ロジックはtrainループ内のevaluateでカバー）
        pass

    def save_graph_data(self, global_graph_dict):
        path = os.path.join(self.experiment_dir, "global_graph.pkl")
        with open(path, 'wb') as f:
            pickle.dump(global_graph_dict, f)
        print(f"グラフデータを保存しました: {path}")

    def save_model_state(self, model, model_name, training_history=None):
        model_path = os.path.join(self.experiment_dir, "models", f"{model_name.replace(' ', '_')}.pth")
        torch.save({
            'model_state_dict': model.state_dict(),
            'training_history': training_history
        }, model_path)
        print(f"モデル保存: {model_path}")

    def save_training_metrics(self, model_name, training_history, final_metrics):
        metrics_path = os.path.join(self.experiment_dir, "metadata", f"training_metrics_{model_name.replace(' ', '_')}.json")
        with open(metrics_path, 'w') as f:
            json.dump({'history': training_history, 'final': final_metrics}, f, indent=2, default=str)
        print(f"学習指標保存: {metrics_path}")

    def generate_summary_report(self, model_name, best_score, training_time):
        report_path = os.path.join(self.experiment_dir, "experiment_summary.md")
        with open(report_path, 'w') as f:
            f.write(f"# Summary\nModel: {model_name}\nBest Score: {best_score}\nTime: {training_time}s")
        print(f"サマリーレポート生成: {report_path}")

# エンコーダ
class SharedVGAEEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels * 2, heads=2, concat=False)
        self.conv2 = GATConv(hidden_channels * 2, hidden_channels, heads=2, concat=False)
        self.conv3 = GATConv(hidden_channels, hidden_channels, heads=1, concat=False)
        self.conv_mu = GATConv(hidden_channels, out_channels, heads=1, concat=False)
        self.conv_logvar = GATConv(hidden_channels, out_channels, heads=1, concat=False)
        self.dropout = nn.Dropout(0.2)
        self.batch_norm1 = nn.BatchNorm1d(hidden_channels * 2)
        self.batch_norm2 = nn.BatchNorm1d(hidden_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.batch_norm1(self.conv1(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.batch_norm2(self.conv2(x, edge_index)))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, edge_index))
        return self.conv_mu(x, edge_index), self.conv_logvar(x, edge_index)

# ODE関数
class ODEFunc(nn.Module):
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 1, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, latent_dim)
        )
        self.scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, t, z):
        t_vec = t.expand(z.size(0), 1)
        z_t = torch.cat([z, t_vec], dim=1)
        dz = self.net(z_t)
        return torch.tanh(self.scale) * dz

class NeuralODEPredictor(nn.Module):
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.ode_func = ODEFunc(latent_dim, hidden_dim)

    def forward(self, z_current, delta_t=0.5):
        t_span = torch.tensor([0., delta_t], device=z_current.device)
        try:
            z_future = odeint(
                self.ode_func,
                z_current,
                t_span,
                method='dopri5',
                rtol=1e-4,
                atol=1e-4,
                options={'max_num_steps': 1000}
            )[-1]
            return z_future
        except Exception as e:
            print(f"ODE予測エラー: {e}")
            return z_current

# =========================================================================
# 2. [改良版] UnifiedVGAE (Inductive & Split Dynamics)
# =========================================================================
class UnifiedVGAE(nn.Module):
    def __init__(self, num_nodes, num_corps, input_dim=128, hidden_dim=64, latent_dim=16):
        super().__init__()
        self.num_nodes = num_nodes
        self.num_corps = num_corps
        self.latent_dim = latent_dim
        self.input_dim = input_dim

        # [修正1] Inductive対応
        # 企業: 固定ID (Transductive)
        self.corp_embeddings = nn.Embedding(num_corps, input_dim)
        nn.init.normal_(self.corp_embeddings.weight, mean=0.0, std=0.05)
        
        # 特許: 特徴量Projection (Inductive)
        self.patent_feature_proj = nn.Sequential(
            nn.Linear(input_dim, input_dim),
            nn.BatchNorm1d(input_dim),
            nn.ReLU(),
            nn.Dropout(0.1)
        )

        self.encoder = SharedVGAEEncoder(input_dim, hidden_dim, latent_dim)

        # [修正2] Split Dynamics (企業と特許で動きを分ける)
        
        self.corp_ode = NeuralODEPredictor(latent_dim, hidden_dim)
        self.patent_ode = NeuralODEPredictor(latent_dim, hidden_dim)

        self.link_predictor = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.generative_decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, input_dim)
        )

    def get_node_features(self, x, node_indices=None):
        if node_indices is None:
            node_indices = torch.arange(self.num_nodes, device=x.device)
            
        out_features = torch.zeros(len(node_indices), self.input_dim, device=x.device)
        corp_mask = node_indices < self.num_corps
        patent_mask = ~corp_mask
        
        if corp_mask.any():
            corp_indices = node_indices[corp_mask]
            out_features[corp_mask] = self.corp_embeddings(corp_indices)
            
        if patent_mask.any():
            # [修正1] 特許は入力xからProjection
            raw_patent_features = x[node_indices[patent_mask]]
            out_features[patent_mask] = self.patent_feature_proj(raw_patent_features)
            
        return out_features

    def decode_features(self, z):
        return self.generative_decoder(z)

    def encode(self, x, edge_index, node_indices=None):
        edge_index = edge_index.long()
        if node_indices is None:
             node_indices = torch.arange(self.num_nodes, device=x.device)
        x_features = self.get_node_features(x, node_indices)
        mu, logvar = self.encoder(x_features, edge_index)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z, edge_index):
        edge_index = edge_index.long()
        z_i = z[edge_index[0]]
        z_j = z[edge_index[1]]
        combined = torch.cat([z_i, z_j], dim=-1)
        return torch.sigmoid(self.link_predictor(combined)).squeeze()

    def predict_future(self, z_current):
        # [修正2] 企業と特許で異なるODEを適用
        z_future = torch.zeros_like(z_current)
        node_indices = torch.arange(z_current.size(0), device=z_current.device)
        
        corp_mask = node_indices < self.num_corps
        patent_mask = ~corp_mask
        
        if corp_mask.any():
            z_future[corp_mask] = self.corp_ode(z_current[corp_mask])
        if patent_mask.any():
            z_future[patent_mask] = self.patent_ode(z_current[patent_mask])
            
        return z_future

# 損失計算ヘルパー
def compute_alignment_loss(z_companies, z_patents):
    if z_companies.size(0) == 0 or z_patents.size(0) == 0:
        return torch.tensor(0.0, device=z_companies.device)
    mean_companies = z_companies.mean(dim=0)
    mean_patents = z_patents.mean(dim=0)
    return F.mse_loss(mean_companies, mean_patents)

# サンプリング関数
def sample_hard_negatives(model, z_t, active_corps, active_patents, pos_set, num_samples=500):
    if len(active_corps) == 0 or len(active_patents) == 0: return None
    # 簡易実装: ランダムサンプリング主体で高速化
    neg_edges = []
    attempts = 0
    while len(neg_edges) < num_samples and attempts < num_samples * 5:
        corp_idx = active_corps[torch.randint(len(active_corps), (1,))].item()
        patent_idx = active_patents[torch.randint(len(active_patents), (1,))].item()
        if (corp_idx, patent_idx) not in pos_set:
            neg_edges.append([corp_idx, patent_idx])
        attempts += 1
    if neg_edges:
        return torch.tensor(neg_edges, dtype=torch.long).t().to(device)
    return None

# [修正3] 損失計算 (KL Annealing対応)
def compute_loss(model, data_t, data_t1, num_corps, beta=0.01):
    try:
        # VAE(t)
        x_features = model.get_node_features(data_t.x)
        mu_t, logvar_t = model.encoder(x_features, data_t.edge_index)
        z_t = model.reparameterize(mu_t, logvar_t)
        
        # Reconstruction Loss
        pos_edge_index = data_t.edge_index
        pos_pred = model.decode(z_t, pos_edge_index)
        pos_loss = -torch.log(pos_pred + 1e-15).mean()
        
        # Negative Sampling
        active_corps = torch.unique(pos_edge_index[0][pos_edge_index[0] < num_corps])
        active_patents = torch.unique(pos_edge_index[1][pos_edge_index[1] >= num_corps])
        pos_set = set(tuple(p.tolist()) for p in pos_edge_index.t())
        neg_edge_index = sample_hard_negatives(model, z_t, active_corps, active_patents, pos_set, num_samples=1000)
        
        neg_loss = torch.tensor(0.0, device=device)
        if neg_edge_index is not None:
            neg_pred = model.decode(z_t, neg_edge_index)
            neg_loss = -torch.log(1 - neg_pred + 1e-15).mean()
        
        recon_loss = pos_loss + neg_loss

        # KL Loss (with Annealing beta)
        kl_loss = -0.5 * torch.mean(1 + logvar_t - mu_t.pow(2) - logvar_t.exp())
        
        # Future Prediction
        z_t1_pred = model.predict_future(mu_t) # ODE
        
        # Future Link Prediction Loss
        pos_edge_index_t1 = data_t1.edge_index
        if pos_edge_index_t1.size(1) > 0:
            future_pos_pred = model.decode(z_t1_pred, pos_edge_index_t1)
            future_pos_loss = -torch.log(future_pos_pred + 1e-15).mean()
            
            # Future Negatives
            active_corps_t1 = torch.unique(pos_edge_index_t1[0][pos_edge_index_t1[0] < num_corps])
            active_patents_t1 = torch.unique(pos_edge_index_t1[1][pos_edge_index_t1[1] >= num_corps])
            pos_set_t1 = set(tuple(p.tolist()) for p in pos_edge_index_t1.t())
            neg_edge_index_t1 = sample_hard_negatives(model, z_t1_pred, active_corps_t1, active_patents_t1, pos_set_t1, num_samples=1000)
            
            future_neg_loss = torch.tensor(0.0, device=device)
            if neg_edge_index_t1 is not None:
                future_neg_pred = model.decode(z_t1_pred, neg_edge_index_t1)
                future_neg_loss = -torch.log(1 - future_neg_pred + 1e-15).mean()
            
            future_loss = future_pos_loss + future_neg_loss
        else:
            future_loss = torch.tensor(0.0, device=device)

        # Total Loss
        total_loss = recon_loss + beta * kl_loss + future_loss
        
        return total_loss, {'recon': recon_loss.item(), 'kl': kl_loss.item(), 'future': future_loss.item()}

    except Exception as e:
        print(f"Loss error: {e}")
        return torch.tensor(0.0, device=device, requires_grad=True), {}

# [修正4] 評価関数 (Recall@K, NDCG@K 追加)
def evaluate_metrics_at_k(model, z_pred, pos_edge_index, k=10):
    
    model.eval()
    with torch.no_grad():
        target_corps = torch.unique(pos_edge_index[0][pos_edge_index[0] < model.num_corps])
        patent_indices = torch.arange(model.num_corps, model.num_nodes, device=device)
        
        recalls = []
        ndcgs = []
        
        # 評価対象企業が多すぎる場合はサンプリング
        if len(target_corps) > 50:
             target_corps = target_corps[torch.randperm(len(target_corps))[:50]]

        for corp_idx in target_corps:
            gt_patents = pos_edge_index[1][pos_edge_index[0] == corp_idx]
            if len(gt_patents) == 0: continue
            
            # この企業と全特許のスコア計算
            z_c = z_pred[corp_idx].unsqueeze(0).expand(len(patent_indices), -1)
            z_p = z_pred[patent_indices]
            combined = torch.cat([z_c, z_p], dim=1)
            scores = torch.sigmoid(model.link_predictor(combined)).squeeze()
            
            # Top-K
            _, top_indices = torch.topk(scores, k)
            top_global_indices = patent_indices[top_indices]
            
            # Recall
            hits = sum([1 for gt in gt_patents if gt in top_global_indices])
            recalls.append(hits / len(gt_patents))
            
            # NDCG (簡易計算)
            dcg = 0.0
            idcg = 0.0
            for i, idx in enumerate(top_global_indices):
                if idx in gt_patents:
                    dcg += 1.0 / np.log2(i + 2)
            for i in range(min(len(gt_patents), k)):
                idcg += 1.0 / np.log2(i + 2)
            ndcgs.append(dcg / idcg if idcg > 0 else 0)
            
    return np.mean(recalls) if recalls else 0, np.mean(ndcgs) if ndcgs else 0

# 学習関数
def train_model(model, global_graph_dict, num_corps, saver, num_epochs=30):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    years = sorted(global_graph_dict.keys())
    
    train_years = years[:-1]
    val_year_t = years[-2]
    val_year_t1 = years[-1]
    
    best_recall = 0
    training_history = {'recall': [], 'ndcg': [], 'losses': []}
    
    print(f"学習開始: {num_epochs} epochs")
    
    for epoch in range(num_epochs):
        model.train()
        
        # KL Annealing (徐々にbetaを上げる)
        beta = min(1.0, (epoch / 10) * 0.1) 
        
        epoch_losses = []
        for i in range(len(train_years) - 1):
            year_t = train_years[i]
            year_t1 = train_years[i+1]
            
            data_t = global_graph_dict[year_t].to(device)
            data_t1 = global_graph_dict[year_t1].to(device)
            
            optimizer.zero_grad()
            loss, loss_dict = compute_loss(model, data_t, data_t1, num_corps, beta=beta)
            
            if loss.item() != 0:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_losses.append(loss.item())
        
        avg_loss = np.mean(epoch_losses) if epoch_losses else 0
        training_history['losses'].append(avg_loss)

        # 検証 (Top-K Metrics)
        if (epoch + 1) % 1 == 0:
            val_data_t = global_graph_dict[val_year_t].to(device)
            val_data_t1 = global_graph_dict[val_year_t1].to(device)
            
            with torch.no_grad():
                x_feat = model.get_node_features(val_data_t.x)
                mu, _ = model.encoder(x_feat, val_data_t.edge_index)
                z_pred = model.predict_future(mu)
                
            recall, ndcg = evaluate_metrics_at_k(model, z_pred, val_data_t1.edge_index, k=10)
            training_history['recall'].append(recall)
            training_history['ndcg'].append(ndcg)
            
            print(f"Epoch {epoch+1}: Loss={avg_loss:.4f} | Val Recall@10={recall:.4f}, NDCG@10={ndcg:.4f}")
            
            if recall > best_recall:
                best_recall = recall
                saver.save_model_state(model, "Best_Unified_Model", training_history)
                
    saver.extract_and_save_latent_vectors(model, global_graph_dict, num_corps, "Unified_Final")
    return model

# メイン実行部
def main():
    print("=" * 60)
    print("Improved Unified VGAE (Inductive + Split ODE)")
    print("=" * 60)

    try:
        cleaned_df = preprocess_data()
        global_graph_dict, all_corporations, all_patents, total_global_nodes, global_corp_to_idx, global_patent_to_idx = build_global_graphs(cleaned_df)

        if global_graph_dict is None:
            return

        num_corps = len(all_corporations)
        
        saver = ModelResultsSaver(output_dir="work/model_outputs")
        saver.save_metadata(all_corporations, all_patents, global_corp_to_idx, global_patent_to_idx, sorted(global_graph_dict.keys()), cleaned_df)

        model = UnifiedVGAE(
            num_nodes=total_global_nodes,
            num_corps=num_corps,
            input_dim=128,
            hidden_dim=64,
            latent_dim=16
        )
        
        start_time = time.time()
        trained_model = train_model(model, global_graph_dict, num_corps, saver, num_epochs=30)
        
        print("Done.")

    except Exception as e:
        print(f"Main Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Using device: cuda
Improved Unified VGAE (Inductive + Split ODE)
データ前処理を開始...
読み込みパス: /home/nakamuraroi/kumagai/work/dataset/kg_dataset.csv
✓ データ前処理完了
グラフ構築: 企業数=2569, 特許数=5753
特許特徴量を準備中...
✓ 24年分のグラフを構築完了
結果は /home/nakamuraroi/kumagai/work/model_outputs/experiment_20251208_120356 に保存されます
メタデータを保存: /home/nakamuraroi/kumagai/work/model_outputs/experiment_20251208_120356/metadata/
学習開始: 30 epochs
Epoch 1: Loss=2.7754 | Val Recall@10=0.0000, NDCG@10=0.0000
Epoch 2: Loss=2.7707 | Val Recall@10=0.0000, NDCG@10=0.0000
Epoch 3: Loss=2.6361 | Val Recall@10=0.0000, NDCG@10=0.0000
Epoch 4: Loss=2.4658 | Val Recall@10=0.0000, NDCG@10=0.0000
Epoch 5: Loss=2.3462 | Val Recall@10=0.0000, NDCG@10=0.0000
Epoch 6: Loss=2.2749 | Val Recall@10=0.0000, NDCG@10=0.0000
Epoch 7: Loss=2.2171 | Val Recall@10=0.0000, NDCG@10=0.0000
Epoch 8: Loss=2.1733 | Val Recall@10=0.0000, NDCG@10=0.0000
Epoch 9: Loss=2.1223 | Val Recall@10=0.0000, NDCG@10=0.0000
Epoch 10: Loss=2.1036 | Val Recall@10=0.0000, NDCG@10=0.0000
E

In [ ]:
kg_df = pd.read_csv(os.path.join(DATA_PATH, "work/dataset/kg_dataset.csv"))
kg_df

,patent_number,patent_name,date,corporation,ipc,lead_ipc,fi,fterm,keyword,description
0,登実第3082540号,Ｚ形仕切付段ボール箱,200106,['佐藤工業株式会社'],[],B65D 5/491 (2006.01),[],['3E060AA03'],"['外装', '表面部', '一部', '外装']",強度が劣る原紙を用いたとしても箱の強度が維持される外装用段ボール箱を得んとするものであり、同...
1,登実第3081641号,吊りフック,200105,"['寄神建設株式会社', '東亜建設工業株式会社', '社団法人日本海上起重技術協会']",[],B66C 1/36 (2006.01),[],['3F004CD08'],"['重量物', '重量物', '遠隔操作', '安全', 'フック']",重量物を吊り下げて設置を行う際の、重量物の開放を遠隔操作で容易かつ安全に行うことのできる吊り...
2,特開2001-311113,橋脚柱の構築工法,200104,['株式会社大林組'],"['E01D 19/02 (2006.01)', 'E01D 21/00 (...",E01D 19/02 (2006.01),"['E01D 19/02', 'E01D 21/04', 'E01D 21/00 B']","['2D059AA03', '2D059CC04']","['作業足場', '型枠材', '組立', '不要', '構築工法']","作業足場や型枠の組立,解体が不要になる橋脚柱の構築工法の提供"
3,特開2001-355221,コンクリートの運搬装置,200104,['株式会社大林組'],['E02B 7/00 (2006.01)'],E02B 7/00 (2006.01),['E02B 7/00 C'],"['3F040BA01', '3F040EA03']",['材料分離'],材料分離と運搬費用の低減
4,特開2001-348842,コンクリートの運搬方法,200104,['株式会社大林組'],['E02B 7/00 (2006.01)'],E02B 7/00 (2006.01),['E02B 7/00 C'],[],['材料分離'],材料分離と運搬費用の低減
...,...,...,...,...,...,...,...,...,...,...
49262,特開2023-112117,試験結果報告方法及び試験結果報告システム,202306,['株式会社大林組'],['G06Q 50/08 (2012.01)'],G06Q 50/08 (2012.01),['G06Q 50/08'],['5L049CC07'],['迅速'],試験場所とは離れた場所で試験結果を確実かつ迅速に確認、承認できるようにするようにする
49263,特開2023-115393,作業支援システム、作業支援方法及び作業支援プログラム,202307,['株式会社大林組'],['G06Q 50/08 (2012.01)'],G06Q 50/08 (2012.01),['G06Q 50/08'],['5L049CC07'],"['揚重作業', '作業現場', '安全']",揚重作業において、作業現場の安全を確保するための作業支援システム、作業支援方法及び作業支援プ...
49264,特開2023-118901,フラッシング水処理用フィルタ、フラッシング水処理用フィルタの製造方法、フラッシング水処理用フ...,202307,['高砂熱学工業株式会社'],['B01D 24/00 (2006.01)'],B01D 24/00 (2006.01),['B01D 29/08 540A'],[],['小型化'],フラッシング排水を処理するフィルタ装置の小型化を図る
49265,特開2023-118908,クリーンルームシステム及び空気循環方法,202307,['高砂熱学工業株式会社'],"['F24F 7/06 (2006.01)', 'F24F 7/007 (...",F24F 7/06 (2006.01),"['F24F 7/06 C', 'F24F 7/007 B', 'F2...",[],"['クリーンルーム', 'エネルギー']",クリーンルームにおけるエネルギーの使用量を少なくする


In [ ]:
import ast
from datetime import datetime
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import textwrap
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import Counter
import hdbscan
import pandas as pd
from sklearn.preprocessing import normalize
from sklearn.feature_extraction.text import TfidfVectorizer
import umap.umap_ as umap

# (★ 新規追加) Neural ODE に必要なライブラリ
import torch
import torch.nn as nn
import torch.optim as optim
try:
    from torchdiffeq import odeint
except ImportError:
    print("="*50)
    print("警告: 'torchdiffeq' が見つかりません。")
    print("Neural ODE の軌跡を描画するには、 'pip install torchdiffeq' を実行してください。")
    print("セクション7は、代わりに単純な軌跡（点と線）を描画します。")
    print("="*50)
    # odeint がなくてもスクリプトが停止しないように、ダミーを定義
    odeint = None
    pass

# === 補助関数 (変更なし) ===
def safe_eval_list(s):
    if isinstance(s, list): return s
    if isinstance(s, str) and s.startswith("[") and s.endswith("]"):
        try: return ast.literal_eval(s)
        except (ValueError, SyntaxError): return []
    return []

def format_yyyymm_jp(date_val):
    if not date_val: return "N/A"
    date_str = str(date_val).strip()
    try:
        dt = datetime.strptime(date_str, "%Y%m")
        return f"{dt.year}年{dt.month}月"
    except ValueError: return date_str

def process_list_string(data):
    if isinstance(data, str) and data.startswith("["):
        try: return eval(data)
        except: return []
    elif isinstance(data, list): return data
    return []

# ===================================================================
# === 1. UMAPによる2次元座標の作成 (変更なし) ===
# ===================================================================
print("UMAPによる2次元への次元削減を開始...")
latent_cols = [col for col in merged_df_after.columns if col.startswith('latent_dim_')]
high_dim_vectors = merged_df_after[latent_cols]
LATENT_DIM = len(latent_cols) # ★ Neural ODE のために次元数を保存
print(f"UMAPの入力ベクトルを準備しました: {high_dim_vectors.shape}")

umap_model = umap.UMAP(
    n_components=2, n_neighbors=15, min_dist=0.1,
    metric='euclidean', random_state=42
)
print("UMAPによる次元削減を実行中...")
embedding = umap_model.fit_transform(high_dim_vectors)
merged_df_after['x'] = embedding[:, 0]
merged_df_after['y'] = embedding[:, 1]
print("✓ 'merged_df_after' に 'x' と 'y' カラムが追加されました。")

# ===================================================================
# === 2. HDBSCANによるクラスタリング (変更なし) ===
# ===================================================================
print("HDBSCANによるクラスタリングを開始...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=15, min_samples=3,
    metric='euclidean', gen_min_span_tree=True, prediction_data=True
)
print("HDBSCANによるクラスタリングを実行中...")
merged_df_after["cluster_id"] = clusterer.fit_predict(high_dim_vectors)
merged_df_after["probability"] = clusterer.probabilities_
print("✓ HDBSCANクラスタリング完了。")

# ===================================================================
# === 3. c-TF-IDF による代表キーワード抽出 (変更なし) ===
# ===================================================================
print("c-TF-IDFによるキーワード抽出を開始...")
cluster_docs = {}
cluster_ids = []
for cid, group in merged_df_after.groupby("cluster_id"):
    if cid == -1: continue
    keywords = []
    for kw in group["processed_keywords"]:
        keywords.extend(process_list_string(kw))
    cluster_docs[cid] = " ".join(keywords)
    cluster_ids.append(cid)

cluster_top_keywords = {}

if cluster_docs:
    doc_list = [cluster_docs[cid] for cid in cluster_ids]
    vectorizer = TfidfVectorizer(min_df=1, stop_words=None, token_pattern=r"(?u)\b\w+\b")
    tfidf_matrix = vectorizer.fit_transform(doc_list)
    feature_names = vectorizer.get_feature_names_out()

    tfidf_scores = tfidf_matrix.toarray()
    top_n = 5
    for i, cid in enumerate(cluster_ids):
        cluster_scores = tfidf_scores[i, :]
        top_indices = np.argsort(cluster_scores)[::-1]
        top_kws = [feature_names[idx] for idx in top_indices[:top_n] if cluster_scores[idx] > 0]
        cluster_top_keywords[cid] = top_kws
    print("✓ c-TF-IDFキーワード抽出完了。")
else:
    cluster_top_keywords = {}
    print("✓ c-TF-IDFキーワード抽出完了。（クラスタなし）")

print("---　全クラスタID・件数・代表キーワード")
cluster_counts = merged_df_after["cluster_id"].value_counts()
for cid, count in cluster_counts.items():
    if cid == -1:
        keywords_str = "N/A (Noise)"
    else:
        kws_list = cluster_top_keywords.get(cid, ["（キーワードなし）"])
        keywords_str = ", ".join(kws_list)
    print(f"Cluster ID: {cid}, 件数: {count}, 代表キーワード: {keywords_str}")

# ===================================================================
# === 4. サブプロットの作成（周辺分布用） (変更なし) ===
# ===================================================================
print("Plotlyフィギュアの準備...")
fig = make_subplots(
    rows=2, cols=2,
    row_heights=[0.15, 0.85],
    column_widths=[0.85, 0.15],
    horizontal_spacing=0.01,
    vertical_spacing=0.01,
    specs=[[{"type": "histogram"}, None],
           [{"type": "scatter"}, {"type": "histogram"}]]
)
fig.add_trace(go.Histogram2dContour(
    x=merged_df_after['x'], y=merged_df_after['y'],
    nbinsx=30, nbinsy=30, colorscale='Blues',
    contours=dict(showlines=False), ncontours=25,
    showscale=False, opacity=0.3, hoverinfo='skip',
), row=2, col=1)

# ===================================================================
# === 5. メインの散布図 (変更なし) ===
# ===================================================================
hover_texts = []
WRAP_WIDTH = 60
print("ホバーテキストの折り返し処理を開始 (c-TF-IDFキーワード統合)...")
for _, row in merged_df_after.iterrows():
    patent_num = row['patent_number']
    title = row.get('patent_name', '')
    filing_date = format_yyyymm_jp(row.get('date', ''))
    lead_ipc = row.get('lead_ipc', '')
    description = str(row.get('description', ''))[:200] + "..."
    cluster_id = row['cluster_id']
    if cluster_id == -1:
        cluster_kws_text = "N/A (Noise)"
    else:
        kws = cluster_top_keywords.get(cluster_id, ["(キーワードなし)"])
        cluster_kws_text = ", ".join(kws)
    corporations = ', '.join(process_list_string(row.get('corporation', '')))
    ipcs = ', '.join(process_list_string(row.get('ipc', '')))
    processed_keywords = ', '.join(process_list_string(row.get('processed_keywords', '')))
    title_wrapped = textwrap.fill(title, WRAP_WIDTH).replace('\n', '<br>')
    corps_wrapped = textwrap.fill(corporations, WRAP_WIDTH).replace('\n', '<br>')
    ipcs_wrapped = textwrap.fill(ipcs, WRAP_WIDTH).replace('\n', '<br>')
    keywords_wrapped = textwrap.fill(processed_keywords, WRAP_WIDTH).replace('\n', '<br>')
    desc_wrapped = textwrap.fill(description, WRAP_WIDTH).replace('\n', '<br>')
    cluster_kws_wrapped = textwrap.fill(cluster_kws_text, WRAP_WIDTH).replace('\n', '<br>')
    hover_text = (
        f"<b>Cluster ID: {cluster_id}</b><br>"
        f"<b>Cluster Keywords:</b> {cluster_kws_wrapped}<br>"
        f"<hr>"
        f"<b>Patent Number:</b> {patent_num}<br>"
        f"<b>Corporation:</b> {corps_wrapped}<br>"
        f"<b>Title:</b> {title_wrapped}<br>"
        f"<b>Filing Date:</b> {filing_date}<br>"
        f"<b>Lead IPC:</b> {lead_ipc}<br>"
        f"<b>IPC:</b> {ipcs_wrapped}<br>"
        f"<b>Keyword:</b> {keywords_wrapped}<br>"
        f"<b>Description:</b> {desc_wrapped}"
    )
    hover_texts.append(hover_text)
merged_df_after['hover_text'] = hover_texts
print(f"ホバーテキストの準備完了 ({len(hover_texts)}件)")
noise_df = merged_df_after[merged_df_after['cluster_id'] == -1]
fig.add_trace(go.Scatter(
    x=noise_df['x'], y=noise_df['y'], mode='markers',
    marker=dict(size=3, color='rgba(200, 200, 200, 0.4)', opacity=0.4),
    text=noise_df['hover_text'], hoverinfo='text', name="Noise (-1)",
    hoverlabel=dict(bgcolor="white", font_size=12, align='left')
), row=2, col=1)
cluster_df = merged_df_after[merged_df_after['cluster_id'] >= 0]
fig.add_trace(go.Scatter(
    x=cluster_df['x'], y=cluster_df['y'], mode='markers',
    marker=dict(
        size=5, color=cluster_df["cluster_id"], colorscale='Viridis',
        opacity=0.7, showscale=False, line=dict(width=0.5, color='white')
    ),
    text=cluster_df['hover_text'], hoverinfo='text', name="Clusters (0+)",
    hoverlabel=dict(bgcolor="white", font_size=12, align='left')
), row=2, col=1)

# ===================================================================
# === 6. 注目企業のハイライト (変更なし) ===
# ===================================================================
print("特定企業のハイライト処理を開始...")
target_companies = ['株式会社大林組']
target_companies_set = set(target_companies)
mask_highlight = merged_df_after['corporation'].apply(
    lambda x: any(c in target_companies_set for c in process_list_string(x))
)
df_highlight = merged_df_after[mask_highlight]
if not df_highlight.empty:
    hover_texts_highlight = df_highlight['hover_text']
    fig.add_trace(go.Scatter(
        x=df_highlight['x'], y=df_highlight['y'], mode='markers',
        marker=dict(
            size=8, color='rgba(255, 0, 0, 0.4)', symbol='circle',
            line=dict(width=1.5, color='red')
        ),
        text=hover_texts_highlight, hoverinfo='text', name=', '.join(target_companies),
        hoverlabel=dict(bgcolor="black", font_color="white", font_size=12, align='left')
    ), row=2, col=1)

# ===================================================================
# === 7. (★修正) Neural ODE軌跡と「軌跡上の」年代マッピング ===
# ===================================================================
print("Neural ODE 連続軌跡の学習・描画を開始...")
TARGET_COMPANY_ODE = '株式会社大林組'

if odeint is None:
    # --- 7.A. (フォールバック) 単純な軌跡 (変更なし) ---
    print(f"--- 'torchdiffeq' が見つからないため、Neural ODE 軌跡 (セクション7) をスキップします ---")
    print(f"--- 代わりに、従来の単純な軌跡（点と線）を描画します ---")
    trajectory_target_companies = ['株式会社大林組']
    trajectory_target_set = set(trajectory_target_companies)
    trajectory_data = []
    for _, row in merged_df_after.iterrows():
        corps_list = process_list_string(row.get('corporation', []))
        found_corps = trajectory_target_set.intersection(corps_list)
        if found_corps:
            try: year = int(str(row['date'])[:4])
            except (ValueError, TypeError): continue
            for corp in found_corps:
                trajectory_data.append({'company': corp, 'year': year, 'x': row['x'], 'y': row['y']})
    if trajectory_data:
        traj_df = pd.DataFrame(trajectory_data)
        avg_positions_df = traj_df.groupby(['company', 'year'])[['x', 'y']].mean().reset_index().sort_values(by='year')
        colors = ['blue']
        fill_colors = ['rgba(0, 0, 255, 0.4)']
        for i, company_name in enumerate(trajectory_target_companies):
            company_data = avg_positions_df[avg_positions_df['company'] == company_name]
            if not company_data.empty:
                fig.add_trace(go.Scatter(
                    x=company_data['x'], y=company_data['y'], mode='lines+markers+text',
                    name=f"{company_name} 軌跡",
                    line=dict(color=colors[i % len(colors)], width=2.5, dash='dot'),
                    marker=dict(size=10, color=colors[i % len(colors)], symbol='diamond'),
                    text=company_data['year'].astype(str), textposition="top right",
                ), row=2, col=1)
                fig.add_trace(go.Scatter(
                    x=company_data['x'], y=company_data['y'], mode='markers',
                    name=f"{company_name} 観測点",
                    marker=dict(
                        size=10,
                        color=fill_colors[i % len(fill_colors)],
                        symbol='diamond',
                        line=dict(width=1.5, color=colors[i % len(colors)])
                    ),
                ), row=2, col=1)

else:
    # --- 7.B. (本命) Neural ODE 軌跡 ---

    # --- 7.1. (★ 修正: 事前計算された年別「企業ベクトル」を読み込む) ---
    print(f"{TARGET_COMPANY_ODE} の「年ごと」の企業ベクトルを読み込み中...")

    # (仮定) ログに基づき、年別ファイルが保存されているパスを指定
    # (★注意) このパスはご自身の環境に合わせて確認してください
    base_path = "/content/drive/MyDrive/model_outputs/experiment_20251105_052244/latent_vectors/after_training_VGAE___ODE/"

    # (重要) セクション1で定義された latent_cols を使用

    all_company_dfs = []

    # ログに基づき、2000年から2023年までループ
    for year in range(2000, 2024):
        filepath = f"{base_path}companies_{year}.csv"
        try:
            df = pd.read_csv(filepath)
            df['year'] = year # 年情報を付与
            all_company_dfs.append(df)
        except FileNotFoundError:
            print(f"警告: {filepath} が見つかりません。スキップします。")
        except Exception as e:
            print(f"エラー: {filepath} の読み込み中に問題発生: {e}")

    if not all_company_dfs:
        print(f"エラー: 年別の企業ベクトルファイル ({base_path}companies_YYYY.csv) が見つかりませんでした。")
        trajectory_data_ode = [] # (元のコードのif not trajectory_data_ode: で検出させる)

    else:
        # 1つのDataFrameに統合
        all_companies_df = pd.concat(all_company_dfs, ignore_index=True)

        # (仮定) 企業名カラムが 'corporation' であると仮定。
        # (★注意) もし 'company_name' など別の名前の場合は、ここを修正してください。
        company_name_column = 'company_name'

        if company_name_column not in all_companies_df.columns:
             print(f"エラー: 企業DataFrameに '{company_name_column}' カラムが見つかりません。")
             print(f"利用可能なカラム: {all_companies_df.columns.tolist()}")
             trajectory_data_ode = [] # エラー
        else:
            # 目的の企業のデータだけを抽出
            company_df = all_companies_df[
                all_companies_df[company_name_column] == TARGET_COMPANY_ODE
            ].sort_values(by='year')

            if company_df.empty:
                print(f"エラー: 読み込んだ企業データ内で {TARGET_COMPANY_ODE} のデータが見つかりません。")
                trajectory_data_ode = [] # エラー
            else:
                # 7.1の処理が成功したことを示す
                trajectory_data_ode = [True] # (空でなければOK)

                # Neural ODE用のTensorに変換 (★平均処理は不要)
                t_values = torch.tensor(company_df['year'].values, dtype=torch.float32)
                z_vectors = torch.tensor(company_df[latent_cols].values, dtype=torch.float32)

                # (★注意) 観測点プロット(7.4)のために年平均(avg_vectors_by_year)の代わりに
                # company_df を直接使うように変更します
                # avg_vectors_by_year = company_df # この変数は後続のコードで使われる

                START_YEAR = int(t_values.min().item())
                END_YEAR = int(t_values.max().item())
                print(f"✓ (ODE) データ準備完了: {START_YEAR}年〜{END_YEAR}年 ({len(t_values)}点)")

    # --- (元のコードの続き) ---
    # 7.1 で失敗した場合 (trajectory_data_ode が空の場合)、ODE処理をスキップ
    if not trajectory_data_ode:
        print(f"エラー: {TARGET_COMPANY_ODE} のデータ（2000-2023）の準備に失敗しました。ODE軌跡をスキップします。")
    else:
        # --- 7.2. (ODE) モデル定義 (変更なし) ---
        class ODEFunc(nn.Module):
            def __init__(self, latent_dim):
                super(ODEFunc, self).__init__()
                self.net = nn.Sequential(
                    nn.Linear(latent_dim, 64), nn.Tanh(),
                    nn.Linear(64, 64), nn.Tanh(),
                    nn.Linear(64, latent_dim),
                )
            def forward(self, t, z):
                return self.net(z)

        # --- 7.3. (ODE) モデル学習 (変更なし) ---
        print(f"Neural ODE の学習を開始します (出発点: {START_YEAR}年)...")
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        ode_func = ODEFunc(LATENT_DIM).to(device)
        optimizer = optim.Adam(ode_func.parameters(), lr=1e-3)
        z_vectors = z_vectors.to(device)
        t_values = t_values.to(device)
        z0_start_vector = z_vectors[0]

        for epoch in range(500):
            optimizer.zero_grad()
            z_pred = odeint(ode_func, z0_start_vector, t_values, method='dopri5')
            loss = torch.mean(torch.abs(z_pred - z_vectors))
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 100 == 0:
                print(f"Epoch {epoch + 1}/{500}, Loss: {loss.item():.6f}")
        print("✓ Neural ODE の学習が完了しました。")

        # --- 7.4. (ODE) 可視化 (変更なし) ---
        # (★注意: company_df から 'year' を取得するように内部で調整)
        print(f"学習した軌跡を投影します (期間: {START_YEAR}年〜{END_YEAR}年)...")
        with torch.no_grad():
            t_dense = torch.linspace(START_YEAR, END_YEAR, 200).to(device)
            z_continuous_high_dim = odeint(ode_func, z0_start_vector, t_dense).cpu().numpy()
            z_pred_high_dim_integers = odeint(ode_func, z0_start_vector, t_values).cpu().numpy()

        z_continuous_2d = umap_model.transform(z_continuous_high_dim)
        z_observed_2d = umap_model.transform(z_vectors.cpu().numpy()) # 観測点
        z_pred_2d_integers = umap_model.transform(z_pred_high_dim_integers)

        # (1) 観測点（●）をプロット
        fig.add_trace(go.Scatter(
            x=z_observed_2d[:, 0],
            y=z_observed_2d[:, 1],
            mode='markers',
            name=f"{TARGET_COMPANY_ODE} (観測点)",
            marker=dict(
                size=10,
                color='rgba(0, 0, 255, 0.4)',
                symbol='circle',
                line=dict(width=1.5, color='blue')
            ),
            # (★修正) avg_vectors_by_year の代わりに company_df['year'] を使用
            text=[f"<b>{int(y)}</b>" for y in company_df['year']],
            legendgroup="neural_ode_traj",
            hoverinfo='none',
        ), row=2, col=1)

        # (2) Neural ODEによる連続曲線（―）をプロット
        fig.add_trace(go.Scatter(
            x=z_continuous_2d[:, 0],
            y=z_continuous_2d[:, 1],
            mode='lines',
            name=f"{TARGET_COMPANY_ODE} (NeuralODE軌跡)",
            line=dict(color='blue', width=2, dash='solid'),
            hoverinfo='none',
            legendgroup="neural_ode_traj",
        ), row=2, col=1)

        # (3) 軌跡上の年代ラベルをプロット
        fig.add_trace(go.Scatter(
            x=z_pred_2d_integers[:, 0],
            y=z_pred_2d_integers[:, 1],
            mode='text',
            name=f"{TARGET_COMPANY_ODE} (年代ラベル)",
             # (★修正) avg_vectors_by_year の代わりに company_df['year'] を使用
            text=[f"<b>{int(y)}</b>" for y in company_df['year']],
            textposition="top right",
            textfont=dict(size=11, color='blue'),
            legendgroup="neural_ode_traj",
            hoverinfo='none',
        ), row=2, col=1)

        print("✓ Neural ODE 軌跡と「軌跡上の年代ラベル」をプロットに追加しました。")

# ===================================================================
# === 8. 円形ガイドライン (変更なし) ===
# ===================================================================
x_min, x_max = merged_df_after['x'].min(), merged_df_after['x'].max()
y_min, y_max = merged_df_after['y'].min(), merged_df_after['y'].max()
x_range = x_max - x_min
y_range = y_max - y_min
max_range = max(x_range, y_range)
margin = max_range * 0.3
x_min -= margin; x_max += margin; y_min -= margin; y_max += margin
center_x = (x_min + x_max) / 2
center_y = (y_min + y_max) / 2
max_radius = np.sqrt(((merged_df_after[['x', 'y']].values - [center_x, center_y])**2).sum(axis=1)).max()
num_circles = 5
for i in range(1, num_circles + 1):
    r = max_radius * (i / num_circles)
    theta = np.linspace(0, 2*np.pi, 300)
    circle_x = center_x + r * np.cos(theta)
    circle_y = center_y + r * np.sin(theta)
    fig.add_trace(go.Scatter(
        x=circle_x, y=circle_y, mode='lines',
        line=dict(color='lightgray', dash='dot', width=1),
        hoverinfo='skip', showlegend=False
    ), row=2, col=1)

# ===================================================================
# === 9. クラスタ中心に代表キーワードラベルを追加 (変更なし) ===
# ===================================================================
print("クラスタラベルの描画をスキップ (ホバーテキストに統合)。")
for cid, group in merged_df_after[merged_df_after['cluster_id'] >= 0].groupby("cluster_id"):
    cx, cy = group["x"].mean(), group["y"].mean()
    label_kws = cluster_top_keywords.get(cid, [])
    label = f"<b>Cluster {cid}</b><br>" + "<br>".join(label_kws)
    fig.add_trace(go.Scatter(
        x=[cx], y=[cy], mode="text", text=[label],
        textfont=dict(size=10, color='#333333', family='Arial'),
        textposition="top center", hoverinfo='skip', showlegend=False
    ), row=2, col=1)

# ===================================================================
# === 10. レイアウト調整 (変更なし) ===
# ===================================================================
fig.update_xaxes(range=[x_min, x_max], showgrid=True, gridcolor='#e0e0e0', zeroline=False, row=2, col=1)
fig.update_yaxes(range=[y_min, y_max], showgrid=True, gridcolor='#e0e0e0', zeroline=False, scaleanchor="x", scaleratio=1, row=2, col=1)
fig.update_xaxes(range=[x_min, x_max], showticklabels=False, showgrid=False, row=1, col=1)
fig.update_xaxes(showticklabels=False, showgrid=False, row=2, col=2)
fig.update_yaxes(range=[y_min, y_max], showticklabels=False, showgrid=False, row=2, col=2)

title_text = f"Tech Landscape Map (HDBSCAN: min_cluster_size={clusterer.min_cluster_size}, metric='{clusterer.metric}')"
fig.update_layout(
    title=dict(text=title_text, font=dict(size=18, color='#333333', family='Arial')),
    template="plotly_white",
    height=900, width=1000,
    font=dict(size=11, family='Arial', color='#333333'),
    showlegend=True,
    plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

print("--- 最終プロットを表示 ---")
fig.show()